# RFI Model Evaluation, Profiling, and Optional Post-processing Tutorial

This is a notebook version of the inference script in an interactive IPython workflow.

It supports:

1. Selecting the RFI model size: `large` or `small`.
2. Loading the correct config, weights, and threshold.
3. Evaluating the model on the RFI test set.
4. Profiling CPU and GPU inference speed.
5. Estimating CPU/GPU power usage.
6. Optionally generating prediction masks.
7. Optionally running RFI post-processing.

Run the notebook cells from top to bottom.

## 1. Imports and logging

This cell imports all required libraries and project modules.  

In [ ]:
from __future__ import annotations

import logging
import subprocess
import threading
import time
from pathlib import Path
from typing import Callable, Dict, Optional, Sequence

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch import Tensor

from pipeline.RFI_usecase.post_processing.profiler import profile_inference
from pipeline.RFI_usecase.post_processing.rfi_postprocessing import process_prediction_folder

from pipeline.RFI_usecase.models.unet import UNet
from pipeline.RFI_usecase.models.unet_small import UNetSmall

from pipeline.RFI_usecase.utils.rc_dataloader import (
    RFI4ChannelDataset,
    collate_pad_validmask,
    rfi_mask_resolver,
    build_transforms_from_cfg,
    load_yaml_config,
)

from pipeline.RFI_usecase.utils.utilities import resize_for_loss
from pipeline.RFI_usecase.train_rfi_large import make_val_cfg_from_train_cfg


# ------------------------- Logging ------------------------- #
LOGGER = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Silence noisy dataloader logs.
logging.getLogger("pipeline.RFI_usecase.utils.rc_dataloader").setLevel(logging.WARNING)
logging.getLogger("RFI4ChannelDataset").setLevel(logging.WARNING)

print("Imports completed.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Choose the model

Recommended starting point:

```python
MODEL_SIZE = "large"
RUN_POSTPROCESSING = True
```

Set `RUN_POSTPROCESSING = False` to skip the dB threshold post-processing step.

In [ ]:
# -------------------------------------------------------------------------
# Main notebook settings
# -------------------------------------------------------------------------

MODEL_SIZE = "large"  # Choose: "large" or "small"

BATCH_SIZE = 2
NUM_WORKERS = 1

# Use None to use the model-specific default threshold.
THR_OVERRIDE = None

# Use None to resize logits/masks/valid mask to mask width.
RESIZE_TO = None

# Profiling fallback power values.
# CPU fallback is based on Intel Core i9-10920X TDP.
FALLBACK_CPU_POWER_WATTS = 165.0

# GPU fallback example for RTX 3090.
FALLBACK_GPU_POWER_WATTS = 350.0

# -------------------------------------------------------------------------
# Optional post-processing settings
# -------------------------------------------------------------------------

RUN_POSTPROCESSING = True

PREDICTION_MASK_DIR = Path(
    "/path/to/rfi_prediction_mask"
)

POSTPROCESSING_OUTPUT_DIR_OVERRIDE = None

THRESHOLD_DB = 5.0
FILTER_MODE = "remove_high_rfi"  # Choose: "keep_high_rfi" or "remove_high_rfi"
FILTER_LEVEL = "patch"           # Choose: "region" (slower, per-region) or "patch" (faster, whole-patch)
SAVE_RESULTS = False              # Set to False to skip saving comparison images (timing only)

REGION_STAT = "p90"              # Choose: "mean", "median", "p90"
BACKGROUND_STAT = "median"      # Choose: "mean", "median", "trimmed_mean"

MIN_REGION_AREA = 20
RING_INNER_ITERS = 3
RING_OUTER_ITERS = 12
TRIM_FRACTION = 0.1

## 3. Model-specific default configuration

The large and small models use different:

- config paths
- weight paths
- default thresholds
- stride multiples
- post-processing output directories

In [ ]:
def get_run_config(model_size: str) -> dict:
    '''
    Return default paths and model-specific settings.

    Large model:
        threshold = 0.404
        stride_multiple = 16

    Small model:
        threshold = 0.325
        stride_multiple = 8
    '''

    test_dir = Path(
        "/media/ubuntu_24_04/data/opensar/range_compressed_scaled/rfi-v2/test2"
    )

    if model_size == "large":
        return {
            "cfg_path": Path(
                ".../pipeline/RFI_usecase/config-rfi-large.yaml"
            ),
            "weights": Path(
                "/media/raid/opensar/artifacts/models/rfi/checkpoints/BS64+lr3e-5+focal0.6/epoch_112.pth"
            ),
            "test_dir": test_dir,
            "thr": 0.404,
            "stride_multiple": 16,
            "input_shape": (4, 1520, 1520),
            "postprocessing_output_dir": Path(
                "/media/raid/opensar/code/backend/data/tmp/rfi_postprocessing_outputs_large_model"
            ),
        }

    if model_size == "small":
        return {
            "cfg_path": Path(
                ".../pipeline/RFI_usecase/config-rfi-kd.yaml"
            ),
            "weights": Path(
                "/media/raid/opensar/artifacts/models/rfi/checkpoints/focal+lr1e-4+bs4+kdfeature/student_epoch_100.pth"
            ),
            "test_dir": test_dir,
            "thr": 0.325,
            "stride_multiple": 8,
            "input_shape": (4, 1520, 1520),
            "postprocessing_output_dir": Path(
                "/media/raid/opensar/code/backend/data/tmp/rfi_postprocessing_outputs_small_kd"
            ),
        }

    raise ValueError(f"Unknown model_size={model_size}. Expected 'large' or 'small'.")


run_cfg = get_run_config(MODEL_SIZE)

CFG_PATH = run_cfg["cfg_path"]
WEIGHTS_PATH = run_cfg["weights"]
TEST_DIR = run_cfg["test_dir"]
THRESHOLD = run_cfg["thr"] if THR_OVERRIDE is None else float(THR_OVERRIDE)
STRIDE_MULTIPLE = run_cfg["stride_multiple"]
INPUT_SHAPE = run_cfg["input_shape"]

POSTPROCESSING_OUTPUT_DIR = (
    run_cfg["postprocessing_output_dir"]
    if POSTPROCESSING_OUTPUT_DIR_OVERRIDE is None
    else Path(POSTPROCESSING_OUTPUT_DIR_OVERRIDE)
)

print("=" * 80)
print(f"Model size: {MODEL_SIZE}")
print(f"Config path: {CFG_PATH}")
print(f"Weights path: {WEIGHTS_PATH}")
print(f"Test dir: {TEST_DIR}")
print(f"Threshold: {THRESHOLD}")
print(f"Stride multiple: {STRIDE_MULTIPLE}")
print(f"Input shape for profiling: {INPUT_SHAPE}")
print(f"Run post-processing: {RUN_POSTPROCESSING}")
print("=" * 80)

## 4. Power measurement helpers

This section provides simple CPU and GPU power estimation utilities.

CPU power:

- First tries Intel RAPL via `/sys/class/powercap/intel-rapl/intel-rapl:0/energy_uj`
- Falls back to `FALLBACK_CPU_POWER_WATTS` if RAPL is unavailable

GPU power:

- Samples `nvidia-smi --query-gpu=power.draw`
- Falls back to `FALLBACK_GPU_POWER_WATTS` if unavailable

In [ ]:
def read_cpu_energy_joules() -> Optional[float]:
    '''
    Read CPU package energy using Intel RAPL.

    The file reports cumulative energy in microjoules.
    This function converts it to joules.
    '''

    rapl_path = Path("/sys/class/powercap/intel-rapl/intel-rapl:0/energy_uj")

    if not rapl_path.exists():
        return None

    try:
        energy_microjoules = float(rapl_path.read_text().strip())
        return energy_microjoules / 1_000_000.0

    except PermissionError:
        LOGGER.warning("CPU RAPL file exists but permission is denied: %s", rapl_path)
        return None

    except Exception as exc:
        LOGGER.warning("Could not read CPU RAPL energy: %s", exc)
        return None


def compute_cpu_power_watts(
    energy_before_j: Optional[float],
    energy_after_j: Optional[float],
    elapsed_seconds: float,
) -> Optional[float]:
    '''
    Compute average CPU power in watts from Intel RAPL energy readings.

    Formula:
        power_watts = energy_used_joules / elapsed_seconds
    '''

    if energy_before_j is None or energy_after_j is None:
        return None

    if elapsed_seconds <= 0:
        return None

    energy_used_j = energy_after_j - energy_before_j

    if energy_used_j < 0:
        return None

    return energy_used_j / elapsed_seconds


def read_gpu_power_watts() -> Optional[float]:
    '''
    Read current GPU power draw using nvidia-smi.
    '''

    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=power.draw",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            check=True,
        )

        lines = result.stdout.strip().splitlines()

        if not lines:
            return None

        return float(lines[0].strip())

    except Exception:
        return None


class PowerSampler:
    '''
    Periodically samples power in a background thread.
    '''

    def __init__(
        self,
        read_power_fn: Callable[[], Optional[float]],
        interval_seconds: float = 0.1,
    ) -> None:
        self.read_power_fn = read_power_fn
        self.interval_seconds = interval_seconds
        self.samples: list[float] = []
        self._stop_event = threading.Event()
        self._thread: Optional[threading.Thread] = None

    def start(self) -> None:
        self.samples.clear()
        self._stop_event.clear()

        self._thread = threading.Thread(
            target=self._sample_loop,
            daemon=True,
        )
        self._thread.start()

    def stop(self) -> None:
        self._stop_event.set()

        if self._thread is not None:
            self._thread.join()

    def _sample_loop(self) -> None:
        while not self._stop_event.is_set():
            power_watts = self.read_power_fn()

            if power_watts is not None:
                self.samples.append(power_watts)

            time.sleep(self.interval_seconds)

    def average_power_watts(self) -> Optional[float]:
        if not self.samples:
            return None

        return sum(self.samples) / len(self.samples)


print("Power helper functions are ready.")

## 5. DataLoader builder

This builds the RFI test DataLoader using:

- RFI4ChannelDataset
- validation transforms
- padding to a stride multiple
- valid masks to ignore padded regions

In [ ]:
def build_loader(
    data_dir: Path,
    tfm_cfg: dict,
    batch_size: int,
    num_workers: int,
    stride_multiple: int,
    ignore_index: int,
    shuffle: bool = False,
) -> torch.utils.data.DataLoader:
    '''
    Build DataLoader for RFI4ChannelDataset.
    '''

    tfms = build_transforms_from_cfg(tfm_cfg, split="val")

    dataset = RFI4ChannelDataset(
        data_dir=str(data_dir),
        use_masks=True,
        mask_resolver=rfi_mask_resolver,
        transform=tfms,
        strict_missing_masks=False,
    )

    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=num_workers > 0,
        collate_fn=lambda batch: collate_pad_validmask(
            batch,
            enforce_stride_multiple=True,
            stride_multiple=stride_multiple,
            bypass_padding=False,
            mask_ignore_index=ignore_index,
            device=torch.device("cpu"),
        ),
    )


print("DataLoader helper is ready.")

## 6. Model loading

This function loads either:

- `UNet` for the large model
- `UNetSmall` for the small model

It also loads the checkpoint and switches the model to evaluation mode.

In [ ]:
def load_model(
    model_size: str,
    model_cfg: dict,
    weights: Path,
    device: torch.device,
) -> torch.nn.Module:
    '''
    Load either the large RFI UNet or the small RFI UNetSmall.
    '''

    if model_size == "large":
        model = UNet(
            in_channels=int(model_cfg.get("in_channels", 4)),
            out_channels=int(model_cfg.get("out_channels", 1)),
            base_channels=int(model_cfg.get("base_channels", 32)),
            depth=int(model_cfg.get("depth", 4)),
            bilinear=bool(model_cfg.get("bilinear", True)),
            dropout=float(model_cfg.get("dropout", 0.0)),
        ).to(device)

    elif model_size == "small":
        model = UNetSmall(
            in_channels=int(model_cfg.get("in_channels", 4)),
            out_channels=int(model_cfg.get("out_channels", 1)),
            base_channels=int(model_cfg.get("base_channels", 16)),
            depth=int(model_cfg.get("depth", 3)),
            bilinear=bool(model_cfg.get("bilinear", True)),
            dropout=float(model_cfg.get("dropout", 0.0)),
        ).to(device)

    else:
        raise ValueError(f"Unknown model_size={model_size}. Expected 'large' or 'small'.")

    state = torch.load(str(weights), map_location=device)

    if isinstance(state, dict) and "state_dict" in state:
        model.load_state_dict(state["state_dict"])
    else:
        model.load_state_dict(state)

    model.eval()
    return model


print("Model loader is ready.")

## 7. Metrics

This computes binary segmentation metrics for one batch:

- Dice
- IoU
- Accuracy
- Precision
- Recall
- Specificity
- TP, FP, FN, TN

The valid mask is used to ignore padded regions.

In [ ]:
@torch.no_grad()
def compute_metrics(
    logits: Tensor,
    mask: Tensor,
    valid: Tensor,
    thr: Optional[float] = 0.99,
    ignore_index: int = 255,
) -> Dict[str, float]:
    '''
    Compute binary segmentation metrics for a batch.
    '''

    if logits.dim() != 4:
        raise ValueError(f"Expected logits shape [B, 1, H, W], got {logits.shape}")

    _, channels, _, _ = logits.shape

    if channels != 1:
        raise ValueError(f"compute_metrics assumes 1-channel logits. Got {channels} channels.")

    if valid.dim() == 3:
        valid = valid.unsqueeze(1)

    if mask.dim() == 4:
        mask = mask.squeeze(1)

    probs = torch.sigmoid(logits)

    valid_mask = mask != ignore_index
    valid_bool = valid_mask & (valid > 0.5).squeeze(1)

    if not valid_bool.any():
        return {
            "dice": 0.0,
            "iou": 0.0,
            "acc": 0.0,
            "precision": float("nan"),
            "recall": float("nan"),
            "specificity": float("nan"),
            "tp": 0,
            "fp": 0,
            "fn": 0,
            "tn": 0,
        }

    threshold = 0.5 if thr is None else float(thr)

    pred = (probs > threshold).to(torch.bool).squeeze(1)
    gt = mask == 1

    pred_flat = pred[valid_bool]
    gt_flat = gt[valid_bool]

    tp = (pred_flat & gt_flat).sum().item()
    fp = (pred_flat & ~gt_flat).sum().item()
    fn = (~pred_flat & gt_flat).sum().item()
    tn = (~pred_flat & ~gt_flat).sum().item()

    total = tp + fp + fn + tn

    acc = (tp + tn) / total if total > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

    has_positive_gt = (tp + fn) > 0

    if not has_positive_gt:
        precision = float("nan")
        recall = float("nan")
        iou = float("nan")
        dice = float("nan")
    else:
        precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0

    return {
        "dice": float(dice),
        "iou": float(iou),
        "acc": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }


print("Metric function is ready.")

## 8. Evaluate the test set

This runs the selected model over the whole test set and aggregates global TP, FP, FN, and TN.

The final metrics are computed globally, not averaged batch by batch.

In [ ]:
@torch.no_grad()
def evaluate_test(
    model_size: str,
    cfg_path: Path,
    weights_path: Path,
    test_dir: Path,
    batch_size: int = 4,
    num_workers: int = 4,
    stride_multiple: int = 16,
    ignore_index: int = 255,
    resize_to: Optional[int] = None,
    thr: Optional[float] = 0.7,
) -> Dict[str, float | int]:
    '''
    Evaluate either RFI large or RFI small model on the test set.
    '''

    cfg = load_yaml_config(str(cfg_path))
    tfm_cfg = load_yaml_config(str(cfg.get("transforms_cfg")))
    tfm_cfg_val = make_val_cfg_from_train_cfg(tfm_cfg)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    loader = build_loader(
        data_dir=Path(test_dir),
        tfm_cfg=tfm_cfg_val,
        batch_size=batch_size,
        num_workers=num_workers,
        stride_multiple=stride_multiple,
        ignore_index=ignore_index,
        shuffle=False,
    )

    model = load_model(
        model_size=model_size,
        model_cfg=cfg.get("model", {}),
        weights=Path(weights_path),
        device=device,
    )

    picked_thr = float(thr if thr is not None else 0.5)

    global_tp = 0
    global_fp = 0
    global_fn = 0
    global_tn = 0

    for batch_idx, batch in enumerate(loader, start=1):
        rc = batch.rc.to(device, non_blocking=True)
        mask = batch.mask.to(device, non_blocking=True)
        valid = batch.valid_mask.to(device, non_blocking=True)

        logits = model(rc)

        if resize_to is None:
            lo, ma, va = resize_for_loss(logits, mask, valid, int(mask.shape[-1]))
        else:
            lo, ma, va = resize_for_loss(logits, mask, valid, int(resize_to))

        metrics = compute_metrics(
            logits=lo,
            mask=ma,
            valid=va,
            thr=picked_thr,
            ignore_index=ignore_index,
        )

        global_tp += metrics["tp"]
        global_fp += metrics["fp"]
        global_fn += metrics["fn"]
        global_tn += metrics["tn"]

        if batch_idx % 10 == 0:
            print(f"Processed {batch_idx} batches...")

    total = global_tp + global_fp + global_fn + global_tn

    acc = (global_tp + global_tn) / total if total > 0 else 0.0
    precision = global_tp / (global_tp + global_fp) if (global_tp + global_fp) > 0 else 0.0
    recall = global_tp / (global_tp + global_fn) if (global_tp + global_fn) > 0 else 0.0
    iou = global_tp / (global_tp + global_fp + global_fn) if (global_tp + global_fp + global_fn) > 0 else 0.0
    dice = (2 * global_tp) / (2 * global_tp + global_fp + global_fn) if (2 * global_tp + global_fp + global_fn) > 0 else 0.0

    output = {
        "model_size": model_size,
        "dice": float(dice),
        "iou": float(iou),
        "acc": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "thr": picked_thr,
    }

    print("\nPerformance metrics:")
    for key, value in output.items():
        print(f"{key}: {value}")

    return output

### Run evaluation

This is the main evaluation cell.

In [ ]:
evaluation_results = evaluate_test(
    model_size=MODEL_SIZE,
    cfg_path=CFG_PATH,
    weights_path=WEIGHTS_PATH,
    test_dir=TEST_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    stride_multiple=STRIDE_MULTIPLE,
    ignore_index=255,
    resize_to=RESIZE_TO,
    thr=THRESHOLD,
)

evaluation_results

## 9. CPU/GPU profiling

This profiles inference latency, throughput, MACs/FLOPs, parameters, and memory.

It also attaches approximate power information:

- CPU power from Intel RAPL if available, otherwise fallback TDP
- GPU power from `nvidia-smi` if available, otherwise fallback value

In [ ]:
def print_profiling_results(title: str, results: dict) -> None:
    '''
    Print profiling results in a consistent format.
    '''

    print(f"\n{title}")
    for key, value in results.items():
        print(f"{key}: {value}")


def run_cpu_gpu_profiling(
    model_size: str,
    model_cfg: dict,
    weights_path: Path,
    input_shape: tuple[int, int, int],
    fallback_cpu_power_watts: Optional[float],
    fallback_gpu_power_watts: Optional[float],
) -> dict:
    '''
    Profile the selected model on CPU and GPU.
    '''

    profiling_summary = {}

    print("\nStarting CPU/GPU profiling...")

    # ------------------------------------------------------------------
    # CPU profiling
    # ------------------------------------------------------------------
    print("\nProfiling CPU...")

    cpu_device = torch.device("cpu")

    cpu_model = load_model(
        model_size=model_size,
        model_cfg=model_cfg,
        weights=weights_path,
        device=cpu_device,
    )

    cpu_energy_before_j = read_cpu_energy_joules()
    cpu_time_before = time.perf_counter()

    cpu_results = profile_inference(
        model=cpu_model,
        input_shape=input_shape,
        device=cpu_device,
        dtype=torch.float32,
        warmup=1,
        iters=3,
        use_channels_last=False,
    )

    cpu_time_after = time.perf_counter()
    cpu_energy_after_j = read_cpu_energy_joules()

    cpu_elapsed_seconds = cpu_time_after - cpu_time_before

    cpu_power_watts = compute_cpu_power_watts(
        energy_before_j=cpu_energy_before_j,
        energy_after_j=cpu_energy_after_j,
        elapsed_seconds=cpu_elapsed_seconds,
    )

    if cpu_power_watts is not None:
        cpu_power_source = "intel_rapl"
    else:
        cpu_power_watts = fallback_cpu_power_watts
        cpu_power_source = "fallback" if fallback_cpu_power_watts is not None else "unavailable"

    cpu_results["power_watts"] = cpu_power_watts
    cpu_results["power_source"] = cpu_power_source

    profiling_summary["cpu"] = cpu_results

    print_profiling_results("CPU profiling results:", cpu_results)

    # ------------------------------------------------------------------
    # GPU profiling
    # ------------------------------------------------------------------
    if not torch.cuda.is_available():
        print("\nGPU profiling skipped: CUDA is not available.")
        return profiling_summary

    print("\nProfiling GPU...")

    gpu_device = torch.device("cuda")

    gpu_model = load_model(
        model_size=model_size,
        model_cfg=model_cfg,
        weights=weights_path,
        device=gpu_device,
    )

    gpu_power_sampler = PowerSampler(
        read_power_fn=read_gpu_power_watts,
        interval_seconds=0.1,
    )

    gpu_power_sampler.start()

    gpu_results = profile_inference(
        model=gpu_model,
        input_shape=input_shape,
        device=gpu_device,
        dtype=torch.float16,
        warmup=5,
        iters=20,
        use_channels_last=True,
    )

    gpu_power_sampler.stop()

    gpu_power_watts = gpu_power_sampler.average_power_watts()

    if gpu_power_watts is not None:
        gpu_power_source = "nvidia-smi"
    else:
        gpu_power_watts = fallback_gpu_power_watts
        gpu_power_source = "fallback" if fallback_gpu_power_watts is not None else "unavailable"

    gpu_results["power_watts"] = gpu_power_watts
    gpu_results["power_source"] = gpu_power_source

    profiling_summary["gpu"] = gpu_results

    print_profiling_results("GPU profiling results:", gpu_results)

    return profiling_summary

### Run profiling

This cell may take longer on CPU, especially for large input shapes such as `(4, 1520, 1520)`.

In [ ]:
cfg = load_yaml_config(str(CFG_PATH))

profiling_results = run_cpu_gpu_profiling(
    model_size=MODEL_SIZE,
    model_cfg=cfg.get("model", {}),
    weights_path=WEIGHTS_PATH,
    input_shape=INPUT_SHAPE,
    fallback_cpu_power_watts=FALLBACK_CPU_POWER_WATTS,
    fallback_gpu_power_watts=FALLBACK_GPU_POWER_WATTS,
)

profiling_results

## 10. Optional: generate prediction masks

This step is needed only if you want to run RFI post-processing.

It saves prediction-only PNG masks:

- `0` = background
- `255` = predicted RFI

Output filename format:

```text
<vv_npy_stem>_pred.png
```

In [ ]:
@torch.no_grad()
def generate_prediction_masks(
    model_size: str,
    cfg_path: Path,
    weights_path: Path,
    test_dir: Path,
    prediction_mask_dir: Path,
    stride_multiple: int = 16,
    ignore_index: int = 255,
    resize_to: Optional[int] = None,
    thr: float = 0.5,
) -> None:
    '''
    Run inference on all VV/VH tile pairs in a folder and save prediction-only PNG masks.
    '''

    cfg = load_yaml_config(str(cfg_path))
    tfm_cfg = load_yaml_config(str(cfg.get("transforms_cfg")))
    tfm_cfg_val = make_val_cfg_from_train_cfg(tfm_cfg)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = load_model(
        model_size=model_size,
        model_cfg=cfg.get("model", {}),
        weights=Path(weights_path),
        device=device,
    )

    prediction_mask_dir = Path(prediction_mask_dir)
    prediction_mask_dir.mkdir(parents=True, exist_ok=True)

    dataset = RFI4ChannelDataset(
        data_dir=str(test_dir),
        use_masks=False,
        mask_resolver=rfi_mask_resolver,
        transform=build_transforms_from_cfg(tfm_cfg_val, split="val"),
        strict_missing_masks=False,
    )

    if not hasattr(dataset, "groups"):
        raise AttributeError(
            "RFI4ChannelDataset does not expose dataset.groups. "
            "This function expects dataset.groups[idx]['vv_path'] to exist."
        )

    print(f"\nGenerating prediction masks for {len(dataset)} samples...")
    print(f"Prediction mask dir: {prediction_mask_dir}")

    saved_count = 0

    for idx in range(len(dataset)):
        sample = dataset[idx]

        batch = collate_pad_validmask(
            [sample],
            enforce_stride_multiple=True,
            stride_multiple=stride_multiple,
            bypass_padding=False,
            mask_ignore_index=ignore_index,
            device=device,
        )

        rc = batch.rc
        valid = batch.valid_mask

        logits = model(rc)

        target_size = int(resize_to) if resize_to is not None else int(valid.shape[-1])

        logits_r = F.interpolate(
            logits,
            size=(target_size, target_size),
            mode="bilinear",
            align_corners=False,
        )

        valid_r = F.interpolate(
            valid.float(),
            size=(target_size, target_size),
            mode="nearest",
        )

        # Crop to the valid region.
        valid_bool = valid_r > 0.5
        valid_2d = valid_bool.squeeze(0).squeeze(0)

        ys, xs = torch.where(valid_2d)

        if ys.numel() == 0:
            y0, y1 = 0, valid_2d.shape[0]
            x0, x1 = 0, valid_2d.shape[1]
        else:
            margin = 2
            y0 = max(int(ys.min().item()) - margin, 0)
            y1 = min(int(ys.max().item()) + 1 + margin, valid_2d.shape[0])
            x0 = max(int(xs.min().item()) - margin, 0)
            x1 = min(int(xs.max().item()) + 1 + margin, valid_2d.shape[1])

        logits_r = logits_r[..., y0:y1, x0:x1]
        valid_r = valid_r[..., y0:y1, x0:x1]

        probs = torch.sigmoid(logits_r)
        pred = (probs > float(thr)).float()
        pred = pred * (valid_r > 0.5).float()

        pred_np = (
            pred.squeeze(0)
            .squeeze(0)
            .detach()
            .cpu()
            .numpy()
            .astype("uint8")
            * 255
        )

        group = dataset.groups[idx]

        if "vv_path" not in group:
            raise KeyError(
                "dataset.groups[idx] does not contain 'vv_path'. "
                f"Available keys: {list(group.keys())}"
            )

        vv_name = Path(group["vv_path"]).stem
        output_path = prediction_mask_dir / f"{vv_name}_pred.png"

        Image.fromarray(pred_np).save(output_path)

        saved_count += 1

        if saved_count % 20 == 0:
            print(f"Saved {saved_count} masks...")

    print(f"\nSaved prediction masks: {saved_count}")
    print(f"Prediction mask dir: {prediction_mask_dir}")

### Generate prediction masks

This cell generates binary prediction masks from the model.

In [ ]:
generate_prediction_masks(
    model_size=MODEL_SIZE,
    cfg_path=CFG_PATH,
    weights_path=WEIGHTS_PATH,
    test_dir=TEST_DIR,
    prediction_mask_dir=PREDICTION_MASK_DIR,
    stride_multiple=STRIDE_MULTIPLE,
    ignore_index=255,
    resize_to=RESIZE_TO,
    thr=THRESHOLD,
)

## 11. Optional: run RFI post-processing

Set `RUN_POSTPROCESSING = False` to skip this step.

### Filter modes

The key option is `FILTER_MODE`:

- `remove_high_rfi`: removes regions whose contrast is above the dB threshold
- `keep_high_rfi`: keeps only regions whose contrast is above the dB threshold

### Filter levels (speed vs accuracy)

The `FILTER_LEVEL` parameter controls the filtering granularity:

- `region`: filters individual connected regions using local background rings
  - **Pros**: More accurate, considers local context for each region
  - **Cons**: Slower, especially for patches with many regions
  
- `patch`: filters the entire patch as a single unit using global statistics
  - **Pros**: Much faster, simpler computation
  - **Cons**: Less precise, uses all non-RFI pixels as background
  
**Recommendation**: Use `patch` for faster processing when you want to quickly filter out entire patches. Use `region` when you need precise per-region filtering.

In [ ]:
if RUN_POSTPROCESSING:
    process_prediction_folder(
        sar_data_dir=TEST_DIR,
        prediction_mask_dir=PREDICTION_MASK_DIR,
        output_dir=POSTPROCESSING_OUTPUT_DIR,
        threshold_db=THRESHOLD_DB,
        region_stat=REGION_STAT,
        background_stat=BACKGROUND_STAT,
        min_region_area=MIN_REGION_AREA,
        ring_inner_iters=RING_INNER_ITERS,
        ring_outer_iters=RING_OUTER_ITERS,
        trim_fraction=TRIM_FRACTION,
        pred_suffix="_pred",
        filter_mode=FILTER_MODE,
        filter_level=FILTER_LEVEL,
        save_results=SAVE_RESULTS,
    )

    print("Post-processing completed.")
    print(f"Output dir: {POSTPROCESSING_OUTPUT_DIR}")

else:
    print("Post-processing skipped because RUN_POSTPROCESSING = False.")

## 12. Typical usage patterns

### Evaluate, profile, generate masks, and run post-processing (default)

```python
MODEL_SIZE = "large"  # or "small"
RUN_POSTPROCESSING = True
SAVE_RESULTS = False  # Don't save images (faster)
FILTER_LEVEL = "patch"  # Faster filtering
```

### Evaluate, profile, and generate masks only (skip post-processing)

```python
MODEL_SIZE = "large"  # or "small"
RUN_POSTPROCESSING = False
```

### Run with per-region filtering and save images

```python
RUN_POSTPROCESSING = True
FILTER_LEVEL = "region"  # More precise
SAVE_RESULTS = True  # Save comparison images
PREDICTION_MASK_DIR = Path("/path/to/prediction_masks")
FILTER_MODE = "remove_high_rfi"
THRESHOLD_DB = 5.0
```

### Override the model threshold

```python
THR_OVERRIDE = 0.5
```

## Notes

- GPU profiling is much faster than CPU profiling for this segmentation model.
- Power values are approximate unless measured directly from Intel RAPL or `nvidia-smi`.
- Default settings prioritize speed: `RUN_POSTPROCESSING=True`, `SAVE_RESULTS=False`, `FILTER_LEVEL="patch"`.
- Prediction masks are always generated; `RUN_POSTPROCESSING` only controls the dB threshold filtering step.